In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os

# Mode (matches the directory name in distribution_data and pmf_data)
mode = "total"

# Répertoires
pmf_dir = f'../data/pmf_data/{mode}/'
dist_dir = f'../data/distribution_data/{mode}/'
output_dir = '../plot/'

# Constantes
k_B = 0.008314  # kJ/(mol*K)
T = 303.15
epsilon = 1e-5

In [ ]:
def free_energy(pKa, charge):
    """Calcule l'énergie libre de référence pour les formes neutres"""
    R = 8.314e-3  # kJ/mol.K
    T = 302.3545798089109  # K
    pH = 7.0
    return -charge * R * T * np.log(10) * (pKa - pH)


def get_ref_per_acid(acid):
    """
    Retourne la correction d'énergie libre pour la forme neutre.
    Source: Platzer et al. (2014)
    pKa: ARG 13.9, LYS 10.34, ASP 3.86, GLU 4.34, HIS 6.45, CYS 8.49, TYR 9.76
    """
    # Pour la ref aux résidus chargés
    neutrals = ['SCHE', 'SCHD', 'SCKN', 'SCRN', 'SCDN', 'SCEN', 'SCC', 'SCY']
    pKas = [6.45, 6.45, 10.34, 13.9, 3.86, 4.34, 8.49, 9.76]
    charges = [1, 1, 1, 1, -1, -1, -1, -1]
    
    if acid in neutrals:
        return free_energy(pKas[neutrals.index(acid)], charges[neutrals.index(acid)])
    return 0


def PMF_residue(acid):
    """
    Charge directement le fichier PMF pré-calculé.
    Fichier: pmf_data/{mode}/{acid}/pmf_{acid}.dat
    Colonnes: z, mean, se  ->  renommées en x, PMF_mean, std_error
    """
    fpath = os.path.join(pmf_dir, acid.lower(), f'pmf_{acid.lower()}.dat')
    df = pd.read_table(fpath, sep=r"\s+")
    df.rename(columns={'z': 'x', 'mean': 'PMF_mean', 'se': 'std_error'}, inplace=True)
    return df


def find_zero_density_limit(acid):
    """
    Trouve la valeur |z| la plus haute où la densité moyenne est 0.
    Lit le fichier summary_{acid}.dat (colonnes: z, mean, se).
    
    Returns:
        float: la limite z en dessous de laquelle les données sont invalides (densité nulle)
    """
    fpath = os.path.join(dist_dir, acid.lower(), f'summary_{acid.lower()}.dat')
    if not os.path.isfile(fpath):
        print(f"Attention: {fpath} non trouvé pour {acid}")
        return 0.0
    
    df = pd.read_table(fpath, sep=r'\s+')
    df['z_abs'] = df['z'].abs()
    # Trouver les |z| < 20 où la densité moyenne est nulle
    zeros = df[(df['mean'] == 0.0) & (df['z_abs'] < 20.0)]['z_abs'].values
    
    if len(zeros) == 0:
        return 0.0
    
    return float(zeros.max())


def get_interval_btw_PMF(ref_data, new_data, correction, z_limit_ref=0.0, z_limit_new=0.0):
    """
    Calcule la différence d'énergie libre ΔG = PMF_new - PMF_ref + correction
    et propage l'erreur: SE = sqrt(SE_ref² + SE_new²)
    
    Args:
        ref_data: DataFrame avec x, PMF_mean, std_error (forme neutre)
        new_data: DataFrame avec x, PMF_mean, std_error (forme chargée)
        correction: correction d'énergie libre
        z_limit_ref: limite z pour la forme de référence (points < z_limit sont exclus)
        z_limit_new: limite z pour la nouvelle forme (points < z_limit sont exclus)
    
    Returns:
        DataFrame avec x, DG_mean, DG_std_error
    """
    # Fusionner sur x
    merged = pd.merge(ref_data, new_data, on='x', suffixes=('_ref', '_new'))
    
    # Filtrer les points en dessous des limites de densité nulle
    z_limit = max(z_limit_ref, z_limit_new)
    if z_limit > 0:
        merged = merged[merged['x'] > z_limit].copy()
    
    # Calcul de ΔG
    merged['DG_mean'] = merged['PMF_mean_new'] - merged['PMF_mean_ref'] + correction
    
    # Propagation de l'erreur: sqrt(SE_ref² + SE_new²)
    merged['DG_std_error'] = np.sqrt(merged['std_error_ref']**2 + merged['std_error_new']**2)
    
    return merged[['x', 'DG_mean', 'DG_std_error']]

In [ ]:
# Définition des acides aminés
aa_name = ['SCRN', 'SCR', 'SCKN', 'SCK', 'SCHE', 'SCHP',
           'SCDN', 'SCD', 'SCEN', 'SCE', 'SCC', 'SCCM', 'SCY', 'SCYM']

aa_symbol = ['ARGN', 'ARG', 'LYSN', 'LYS', 'HSE', 'HIS',
             'ASPN', 'ASP', 'GLUN', 'GLU', 'CYSN', 'CYS', 'TYRN', 'TYR']

aa_charge = [0, 1, 0, 1, 0, 1, 0, -1, 0, -1, 0, -1, 0, -1]


def pKa_func(DG, charge):
    """Convertit une différence d'énergie libre en pKa"""
    R = 8.314e-3  # kJ/mol.K
    T = 302.3545798089109  # K
    pH = 7.0
    return -DG / (charge * R * T * np.log(10)) + pH


def compute_pKa_dataframe(filter_plateau=False):
    """
    Calcule les pKa pour tous les acides aminés à partir des fichiers PMF pré-calculés.
    
    Args:
        filter_plateau: Si True, retire les points où la densité est nulle
                        (déterminé depuis les fichiers de distribution)
    
    Returns:
        DataFrame avec x, pKa et std pour chaque acide aminé
    """
    results_list = []
    
    for i in range(0, len(aa_name), 2):
        # Charger les PMF pré-calculés
        ref_data = PMF_residue(aa_name[i])      # Forme neutre
        new_data = PMF_residue(aa_name[i + 1])  # Forme chargée
        
        acid_s = aa_symbol[i + 1]
        charge = aa_charge[i + 1]
        correction = get_ref_per_acid(aa_name[i])
        
        # Trouver les limites z où la densité devient nulle
        if filter_plateau:
            z_limit_ref = find_zero_density_limit(aa_name[i])
            z_limit_new = find_zero_density_limit(aa_name[i + 1])
        else:
            z_limit_ref = 0.0
            z_limit_new = 0.0
        
        # Calcul de ΔG avec propagation d'erreur
        DG = get_interval_btw_PMF(ref_data, new_data, correction, 
                                   z_limit_ref=z_limit_ref, z_limit_new=z_limit_new)
        
        # Conversion en pKa
        DG['pKa'] = pKa_func(DG['DG_mean'], charge)
        # Propagation de l'erreur pour pKa: |dpKa/dDG| * SE_DG
        R = 8.314e-3
        T = 302.3545798089109
        DG['pKa_std'] = np.abs(1 / (charge * R * T * np.log(10))) * DG['DG_std_error']
        
        # Renommer les colonnes pour cet acide
        DG_renamed = DG[['x', 'pKa', 'pKa_std']].rename(columns={
            'pKa': f'pKa_{acid_s}',
            'pKa_std': f'std_{acid_s}'
        })
        results_list.append(DG_renamed)
    
    # Fusionner tous les résultats sur x
    result = results_list[0]
    for df in results_list[1:]:
        result = pd.merge(result, df, on='x', how='outer')
    
    return result.sort_values('x').reset_index(drop=True)


# Calcul des deux DataFrames
print("Calcul des pKa (tous points)...")
df_e = compute_pKa_dataframe(filter_plateau=False)  # Tous les points (avec plateau)

print("Calcul des pKa (sans plateau - basé sur densité nulle)...")
df_0 = compute_pKa_dataframe(filter_plateau=True)   # Sans les points au plateau

# Affichage des limites trouvées
print("\n=== Limites de densité nulle trouvées ===")
for i in range(0, len(aa_name), 2):
    z_ref = find_zero_density_limit(aa_name[i])
    z_new = find_zero_density_limit(aa_name[i + 1])
    print(f"{aa_symbol[i+1]}: ref ({aa_name[i]}) z_min={z_ref:.1f} Å, new ({aa_name[i+1]}) z_min={z_new:.1f} Å")

print("\n=== Résumé des calculs ===")
print("PMF chargés depuis fichiers .dat pré-calculés")
print("Erreur standard: sqrt(SE_ref² + SE_new²)")

print("\nExtrait pour z ∈ [0, 20] Å (tous points):")
display(df_e[(df_e['x'] > 0) & (df_e['x'] < 20)])

print("\nExtrait pour z ∈ [0, 20] Å (sans plateau):")
display(df_0[(df_0['x'] > 0) & (df_0['x'] < 20)])

In [ ]:
# Plot des pKa
color = plt.rcParams['axes.prop_cycle'].by_key()['color']
plt.figure(figsize=(4, 3))
list_aa = ['ARG', 'LYS', 'TYR', 'CYS', 'HIS', 'GLU', 'ASP']

for i, name in enumerate(list_aa):
    # Données sans plateau (ligne pleine)
    mask_0 = df_0['x'].notna() & df_0[f'pKa_{name}'].notna()
    x1 = df_0.loc[mask_0, 'x']
    y1 = df_0.loc[mask_0, f'pKa_{name}']
    e1 = df_0.loc[mask_0, f'std_{name}']
    
    # Données avec plateau (ligne pointillée)
    mask_e = df_e['x'].notna() & df_e[f'pKa_{name}'].notna()
    x2 = df_e.loc[mask_e, 'x']
    y2 = df_e.loc[mask_e, f'pKa_{name}']
    e2 = df_e.loc[mask_e, f'std_{name}']
    
    plt.errorbar(x1, y1, e1, linewidth=1.5, color=color[i], elinewidth=0.4, capthick=0.4, capsize=1)
    plt.errorbar(x2, y2, e2, linewidth=1, color=color[i], linestyle='dashed', alpha=0.5)
    
    # Label à droite
    if len(y1) > 0:
        # Trouver la valeur à z ≈ 40 Å pour positionner le texte
        idx_40 = (x1 - 40).abs().idxmin() if len(x1) > 0 else None
        if idx_40 is not None and idx_40 in y1.index:
            y_pos = y1.loc[idx_40]
            if name not in ['ASP', 'TYR', 'CYS']:
                plt.text(35, y_pos + 0.2, name, weight="bold", color=color[i])
            else:
                plt.text(35, y_pos - 0.7, name, weight="bold", color=color[i])

# Grille verticale personnalisée (lignes à 9.5, 19.5, 30) - en dehors de la boucle
custom_vgrid = [9.5, 19.5, 30]
for xv in custom_vgrid:
    plt.axvline(x=xv, linestyle='--', alpha=0.8, linewidth=1, color='gray', zorder=0)

# Grille horizontale
plt.gca().yaxis.grid(True, which='major', linestyle='--', alpha=0.5, linewidth=0.3)

# Chiffres romains en haut pour les zones membranaires
roman_numerals = [('I', 4.75), ('II', 14.5), ('III', 24.75), ('IV', 35)]
for numeral, x_pos in roman_numerals:
    plt.text(x_pos, 1.03, numeral, transform=plt.gca().get_xaxis_transform(),
             ha='center', va='bottom', fontsize=10, fontweight='normal')

plt.xlim([0, 40])
plt.ylim([3, 16])
plt.xlabel('z ($\AA$)')
plt.ylabel('pKa')
plt.savefig(output_dir + 'pKa.png', bbox_inches='tight', transparent=True, dpi=800)
plt.show()